# Orange Visual Condition Classification — Greenpod Labs Assignment 2 Adaptation

**Classify `blackspot`, `canker`, `fresh`, `grenning` with two distinct models:** HSV/LAB/LBP + SVM and a CNN trained from scratch. This notebook runs the same evaluated pipeline as the Python project.

**Important scope difference:** The company requested *avocado ripening stages*. The uploaded dataset is **orange visual-condition categories**, so this notebook is an orange-only adaptation and does **not** fulfill the fruit/class requirement of the original question.

The reported local reference experiment used 1,090 images (991 provided `train`, 99 provided `test`). It achieved held-out accuracy **94.95%** for SVM and CNN. Your newly trained notebook outputs may vary.

## Step 1 — Upload two ZIP files

Upload **the orange project ZIP** provided in the ChatGPT answer and the original **`Orange Dataset.zip`** file. Colab will prompt you for files. The dataset is not bundled inside the project ZIP to keep the project download small.

In [ ]:
from google.colab import files
from pathlib import Path
uploaded = files.upload()  # Select: OrangeScope_Flask_Clean_Submission.zip AND Orange Dataset.zip
print(list(uploaded))

## Step 2 — Extract the project and data

The Orange Dataset.zip preserves the original `train/` and `test/` split. Its test images must **not** be mixed into training.

In [ ]:
import zipfile, os
root = Path('/content')
project_zip = root/'OrangeScope_Flask_Clean_Submission.zip'
data_zip = root/'Orange Dataset.zip'
assert project_zip.exists(), 'Upload OrangeScope_Flask_Clean_Submission.zip in Step 1'
assert data_zip.exists(), 'Upload Orange Dataset.zip in Step 1'
with zipfile.ZipFile(project_zip) as z: z.extractall(root)
project = root/'orange_disease_classification'
(project/'data'/'raw').mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(data_zip) as z: z.extractall(project/'data'/'raw')
print('Project extracted:',project)
print('Class folders:', sorted(str(x.relative_to(project)) for x in (project/'data'/'raw').glob('train/*')))

## Step 3 — Install dependencies and run unit tests

In [ ]:
%cd /content/orange_disease_classification
%pip -q install 'numpy>=1.24,<3' 'pandas>=2' 'opencv-python-headless>=4.8' 'scikit-image>=0.21' 'scikit-learn>=1.3' 'seaborn>=0.12' 'reportlab>=4' 'joblib>=1.3'
!python -m pip install -q Flask
!python -m unittest discover -s tests -v

## Step 4 — Train both models, audit the dataset and evaluate

This command produces EDA checks, duplicates/brightness checks, split manifest, SVM checkpoint, CNN checkpoint, validation metrics and final held-out test metrics. **The model is selected using validation macro F1; test is reserved for the end.**

On a free Colab CPU this can take some time. Turn on the GPU in *Runtime → Change runtime type* if available.

In [ ]:
!python -u run_pipeline.py --data-root data/raw --output . --architecture cnn --epochs 10 --patience 4 --image-size 96 --batch-size 32

## Step 5 — Inspect real data understanding and held-out results

In [ ]:
import pandas as pd
from IPython.display import display, Image
print('Split sizes:')
display(pd.read_csv('results/split_distribution.csv',index_col=0))
print('Measured model comparison:')
display(pd.read_csv('results/model_comparison.csv'))
display(Image(filename='results/class_balance.png'))
display(Image(filename='results/svm_test_confusion_matrix.png'))
display(Image(filename='results/cnn_test_confusion_matrix.png'))

## Step 6 — Demonstrate one example run with BOTH models

In [ ]:
!python predict.py --image examples/orange_blackspot_example.jpg --expected blackspot --models-dir models --results-dir results
from IPython.display import Image, display
display(Image(filename='results/example_prediction.jpg'))

## Step 7 — Inspect the report and download outputs

In [ ]:
from google.colab import files
from pathlib import Path
report = Path('report/Orange_Condition_Classification_Report.pdf')
assert report.exists()
print('Final report:', report.resolve(), 'bytes:', report.stat().st_size)
files.download(str(report))

## Optional — Full model files and results backup

In [ ]:
import shutil
backup = shutil.make_archive('/content/orange_training_results','zip',root_dir='/content/orange_disease_classification',base_dir='results')
print(backup)
# files.download(backup)  # Uncomment when you need all figures and result tables.

## How to discuss limitations

- The four classes are **orange appearance/condition** categories. They are not *ripening stages*.
- Hand-engineered SVM is computationally simple and interpretable; CNN learns its own patterns but may overfit.
- Background shortcuts are plausible: some `fresh` images use white staged backdrops, whereas some `grenning` images have dark backgrounds.
- Labels use the dataset spelling `grenning` exactly.
- Final test numbers cannot establish production performance. Real deployments require fruit-level/group-independent testing, new camera/orchard data and hardware latency measurements.
- The green candidate box in the demo is not a learned object-detection model.

All four mandatory question answers are in `INTERVIEW_ANSWERS.md` and the PDF report.